# Phase 2 — Step 6: **knowledge distillation** into the small encoder (experiment 4)

**Step 5 results** (fold-2 macro-F1 / micro-F1, BiLSTM head everywhere):

| model                        | method           | trainable params | macro     | micro     |
| ---------------------------- | ---------------- | ---------------- | --------- | --------- |
| A (plain decoder), causal    | **LoRA** r=8     | 2.7M             | **0.509** | **0.622** |
| A2 (decoder+copy), causal    | full fine-tuning | 71.9M            | 0.506     | 0.613     |
| B (prefix-LM), causal        | **LoRMA** r=8    | 2.9M             | 0.503     | 0.616     |
| B (prefix-LM), bidirectional | LoRMA r=8        | 2.9M             | 0.485     | 0.601     |

→ Parameter-efficient methods (≈4% of the weights trainable) **match** full fine-tuning on this small dataset.

**Distillation setup**

- **Teacher**: A2 + full fine-tuning + BiLSTM head (70M backbone, 10 layers).
- **Student**: the **encoder** of the Phase-1 encoder–decoder model D (4 bidirectional layers, ≈40M incl. embeddings) + BiLSTM head.
  D's encoder only saw inputs of ≤ 1024 tokens in Phase 1 → windows of **1024** tokens (stride 768).
- **Loss**: `(1−α)·CE(student, gold) + α·T²·KL(teacher_T ‖ student_T)` with α = 0.5, T = 2.
  The softened teacher distribution also teaches the student the teacher's "second choices".
- **Control run**: the same student trained **without** the teacher (gold labels only) — the difference is the effect of KD.
- **Efficiency**: parameters, inference speed (tokens/s) and memory of teacher vs student.

Teacher outputs are computed once per stage and cached (`teacher_fold1.npy`, `teacher_full.npy`). Everything is cached.
Both students get a Stage B + submission (KD = submission 5, control = submission 6). ≈ 20–30 min.


## 0. Setup


In [1]:
import sys, importlib, json
from pathlib import Path
import torch

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / ".gitignore").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Project root with .gitignore not found")
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
import ner_utils as nu, ner_models as nm, ner_experiment as ne
for m in (nu, nm, ne):
    importlib.reload(m)

TEACHER = "A2_full_causal_bilstm_norm_hm0"
D_RUN = "D_encdec_copy1_e4d5_640_sc70tr30_200M"
STUDENT = {"phase1_run": D_RUN, "causal": False, "head": "bilstm", "max_len": 1024, "stride": 768}
results = {}

c:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Teacher: inference cost (its fold-2 score is known: 0.506)


In [2]:
exp = ne.Experiment(PROJECT_ROOT, {"head": "bilstm"})
results["teacher (A2 full FT)"] = {**exp.speed(), "macro_f1": json.load(open(exp.dir / "val_scores.json"))["val"]["macro_f1"]}
del exp; torch.cuda.empty_cache()

experiment: A2_full_causal_bilstm_norm_hm0 | device cuda | precision torch.bfloat16
parameters: 71.91M total, 71.91M trainable | loaded {'arch': 'A2', 'layers': 10, 'loaded_tensors': 72, 'dropped_tensors': 5, 'dropped_groups': ['copy_gate', 'copy_k', 'copy_q', 'lm_head']}
[cache] training 'fold1' already done -> loaded best_fold1.pt (best epoch 9)
Stage A: best epoch 9 | val macro-F1 0.5057 | 0s


speed: {'tokens_per_sec': 64071, 'infer_sec_fold2': 5.35, 'peak_gb_infer': 0.72, 'params_total': 71906969, 'params_trainable': 71906969}


## 2. Student WITHOUT distillation (control): D encoder + full fine-tuning on gold labels only


In [3]:
exp = ne.Experiment(PROJECT_ROOT, {**STUDENT, "method": "full"})
exp.stage_a()
sc = exp.report(n_examples=0)
results["student, no KD"] = {**exp.speed(), "macro_f1": sc["macro_f1"]}
exp.log()
zip_ctrl = exp.stage_b()
print("UPLOAD (submission 6):", zip_ctrl)
del exp; torch.cuda.empty_cache()

experiment: D_full_bidir_bilstm_norm_hm0_w1024 | device cuda | precision torch.bfloat16
parameters: 42.16M total, 42.16M trainable | loaded {'arch': 'D_encdec', 'layers': 4, 'loaded_tensors': 30, 'dropped_tensors': 61, 'dropped_groups': ['copy_gate', 'copy_k', 'copy_q', 'dec', 'dec_norm', 'lm_head']}


epoch  1 | train loss 1.3511 | 20s | val macro-F1 0.0585 micro-F1 0.1560  <- best


epoch  2 | train loss 0.1552 | 20s | val macro-F1 0.3237 micro-F1 0.5028  <- best


epoch  3 | train loss 0.0708 | 19s | val macro-F1 0.3822 micro-F1 0.5438  <- best


epoch  4 | train loss 0.0374 | 19s | val macro-F1 0.4239 micro-F1 0.5448  <- best


epoch  5 | train loss 0.0224 | 19s | val macro-F1 0.3729 micro-F1 0.5079


... (12 outputs left out)

epoch  5 | train loss 0.0136 | 46s


epoch  6 | train loss 0.0070 | 45s


epoch  7 | train loss 0.0045 | 45s
training 'full' finished -> loaded best epoch 7


submission OK: c:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project\checkpoints\phase2\D_full_bidir_bilstm_norm_hm0_w1024\submission\submission.zip | 35 documents | 4,067 entities
test entities per label: {'STAT': 839, 'DATE': 726, 'PREC': 645, 'COURT': 591, 'CASENO': 299, 'AUTH': 271, 'APP': 219, 'R.COUNSEL': 146, 'JUDGE': 107, 'RESP': 90, 'A.COUNSEL': 70, 'WIT': 64}
UPLOAD (submission 6): c:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project\checkpoints\phase2\D_full_bidir_bilstm_norm_hm0_w1024\submission\submission.zip


## 3. Student WITH distillation from the A2 teacher


In [4]:
exp = ne.Experiment(PROJECT_ROOT, {**STUDENT, "method": "kd", "kd_teacher": TEACHER})
exp.stage_a()
sc = exp.report(n_examples=0)
results["student + KD"] = {**exp.speed(), "macro_f1": sc["macro_f1"]}
exp.log()
zip_kd = exp.stage_b()
print("UPLOAD (submission 5):", zip_kd)
del exp; torch.cuda.empty_cache()

experiment: D_kd_bidir_bilstm_norm_hm0_w1024 | device cuda | precision torch.bfloat16
parameters: 42.16M total, 42.16M trainable | loaded {'arch': 'D_encdec', 'layers': 4, 'loaded_tensors': 30, 'dropped_tensors': 61, 'dropped_groups': ['copy_gate', 'copy_k', 'copy_q', 'dec', 'dec_norm', 'lm_head']}


epoch  1 | train loss 3.4155 | 24s | val macro-F1 0.0012 micro-F1 0.0030  <- best


epoch  2 | train loss 0.4455 | 22s | val macro-F1 0.2532 micro-F1 0.4947  <- best


epoch  3 | train loss 0.1917 | 23s | val macro-F1 0.3017 micro-F1 0.5076  <- best


epoch  4 | train loss 0.1075 | 23s | val macro-F1 0.3771 micro-F1 0.5226  <- best


epoch  5 | train loss 0.0660 | 22s | val macro-F1 0.3592 micro-F1 0.5259


... (18 outputs left out)

epoch  8 | train loss 0.0120 | 44s


epoch  9 | train loss 0.0100 | 45s


epoch 10 | train loss 0.0091 | 47s
training 'full' finished -> loaded best epoch 10


submission OK: c:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project\checkpoints\phase2\D_kd_bidir_bilstm_norm_hm0_w1024\submission\submission.zip | 35 documents | 4,007 entities
test entities per label: {'STAT': 830, 'DATE': 726, 'PREC': 677, 'COURT': 584, 'CASENO': 304, 'AUTH': 272, 'APP': 189, 'R.COUNSEL': 103, 'JUDGE': 102, 'A.COUNSEL': 85, 'RESP': 83, 'WIT': 52}
UPLOAD (submission 5): c:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project\checkpoints\phase2\D_kd_bidir_bilstm_norm_hm0_w1024\submission\submission.zip


## 4. Teacher vs students: accuracy and cost


In [5]:
print(f"{'model':22s} {'macro-F1':>8s} {'params':>9s} {'tokens/s':>9s} {'peak GB':>8s}")
for name, r in results.items():
    print(f"{name:22s} {r['macro_f1']:8.4f} {r['params_total'] / 1e6:8.1f}M {r['tokens_per_sec']:9,d} {str(r['peak_gb_infer']):>8s}")
json.dump(results, open(PROJECT_ROOT / "experiments" / "phase2" / "kd_comparison.json", "w"), indent=1)

model                  macro-F1    params  tokens/s  peak GB
teacher (A2 full FT)     0.5057     71.9M    64,071     0.72
student, no KD           0.4533     42.2M   112,375     0.45
student + KD             0.4319     42.2M    99,302     0.45


## 5. All Phase-2 runs (fold-2 validation)


In [6]:
rows = ne.compare(PROJECT_ROOT)

run                                         macro  micro  ep     APP   RESP A.COUN R.COUN  JUDGE    WIT   AUTH  COURT   STAT   PREC   DATE CASENO
A_lorar8_causal_bilstm_norm_hm0            0.5092 0.6215  15   0.206  0.177  0.357  0.547  0.634  0.366  0.437  0.816  0.735  0.408  0.946  0.480
A2_full_causal_bilstm_norm_hm0             0.5057 0.6133   9   0.208  0.185  0.454  0.505  0.690  0.245  0.361  0.814  0.708  0.476  0.948  0.473
B_lormar8_causal_bilstm_norm_hm0           0.5034 0.6164  13   0.235  0.115  0.438  0.532  0.548  0.335  0.439  0.824  0.724  0.390  0.933  0.528
A2_full_causal_norm_hm0                    0.4975 0.5934   4   0.249  0.215  0.494  0.503  0.651  0.232  0.389  0.784  0.676  0.429  0.941  0.407
B_lormar8_bidir_bilstm_norm_hm0            0.4845 0.6006  10   0.202  0.098  0.271  0.565  0.608  0.281  0.431  0.812  0.711  0.417  0.934  0.485
A2_full_causal_look4_norm_hm0              0.4698 0.5928  12   0.141  0.162  0.335  0.464  0.591  0.245  0.373  0.810  0.672

## After it finishes

Upload the two zips printed as `UPLOAD:` (KD student = submission 5, control = submission 6) and tell me "done" + any Codabench scores.
